In [1]:
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def synthetic_data(w, b, num_examples):
    # 构造随机数据y = Xw + b + e
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = X @ w + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y

def data_iter(batch_size, features, labels):
    # 小批量读取数据
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices) # 原地打乱列表
    for i in range(0, num_examples, batch_size):
        batch_indices = np.array(indices[i:min(i+batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

def linreg(X, w, b):
    # 线性回归模型
    return X @ w + b

def squared_loss(y_hat, y):
    # 均方损失
    return (y_hat - y.reshape(-1,1)) ** 2 / 2

def sgd(params, lr, batch_size):
    # 小批量随机梯度下降
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

        


    

In [3]:
### 参数设定
batch_size = 20
num_examples = 1000
### 参数设定完成

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, num_examples)
#plt.scatter(features[:,1], labels)
#plt.show()

#batch_features, batch_labels = next(data_iter(batch_size, features, labels))
#print(batch_features, batch_labels)

#for X, y in data_iter(batch_size, features, labels):
#    print(X, '\n', y)

In [4]:
### 训练部分
### 参数设定
lr = 0.03
num_epochs = 5
net = linreg
loss = squared_loss

w = torch.normal(0, 0.01, size=(2,1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
### 参数设定完成

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)
        l.sum().backward()
        sgd([w, b], lr, batch_size)
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):.6f}')

print(f'w的估计误差: {true_w - w.reshape(true_w.shape)}')
print(f'b的估计误差: {true_b - b}')




epoch 1, loss 0.787227
epoch 2, loss 0.037335
epoch 3, loss 0.001805
epoch 4, loss 0.000135
epoch 5, loss 0.000056
w的估计误差: tensor([ 0.0007, -0.0021], grad_fn=<SubBackward0>)
b的估计误差: tensor([0.0028], grad_fn=<RsubBackward1>)
